In [1]:
import argparse
import csv
import multiprocessing
import os
import sys
from glob import glob
from typing import Dict, Optional

from loguru import logger

import pyrosetta

ScoreBreakdown = Dict[str, float]

pyrosetta.init(
    " ".join(
        [
            "-mute",
            "all",
            "-use_input_sc",
            "-ignore_unrecognized_res",
            "-ignore_zero_occupancy",
            "false",
            "-load_PDB_components",
            "false",
            "-relax:default_repeats",
            "2",
            "-no_fconfig",
            "-use_terminal_residues",
            "true",
            "-in:file:silent_struct_type",
            "binary",
        ]
    ),
    silent=True,
)

scorefxn = pyrosetta.create_score_function("ref2015")

def _binding_energy(protein_pose, peptide_pose) -> ScoreBreakdown:
    """
    Compute complex binding energy: score(complex) - score(protein) - score(peptide).
    """
    complex_pose = protein_pose.clone()
    complex_pose.append_pose_by_jump(peptide_pose.clone(), complex_pose.total_residue())
    complex_score = scorefxn(complex_pose)
    peptide_score = scorefxn(peptide_pose)
    protein_score = scorefxn(protein_pose)
    return {
        "complex": complex_score,
        "protein": protein_score,
        "peptide": peptide_score,
        "binding": complex_score - (protein_score + peptide_score),
    }

def _binding_energy_from_complex_pose(pose, peptide_chain_id: str):
    """
    Compute binding energy for a complex pose by splitting receptor / peptide chains.
    """
    peptide_pose = pose.clone()
    receptor_pose = pose.clone()
    for idx in range(pose.total_residue(), 0, -1):
        chain = pose.pdb_info().chain(idx)
        if chain == peptide_chain_id:
            receptor_pose.delete_residue_slow(idx)
        else:
            peptide_pose.delete_residue_slow(idx)
    return _binding_energy(receptor_pose, peptide_pose)

def score_complex(pdb_path: str, peptide_chain_id: str='B') -> Optional[ScoreBreakdown]:
    """
    Computes binding energy breakdown for a complex at `pdb_path`.
    """
    try:
        pose = pyrosetta.pose_from_pdb(pdb_path)
        return _binding_energy_from_complex_pose(pose, peptide_chain_id)
    except Exception as exc:
        logger.exception(f"Scoring failed for {pdb_path}: {exc}")
        return None

In [2]:
complexes = os.listdir('./batch_relax')
comp_single = os.listdir('./single_relax')
complexes.sort()
comp_single.sort()

assert len(complexes) == len(comp_single), "Mismatch in number of complexes and singles"

comp_batch = [f'./batch_relax/{f}' for f in complexes]
comp_single = [f'./single_relax/{f}' for f in comp_single]

In [3]:
for i in range(len(comp_batch)):
    batch_score = score_complex(comp_batch[i])['binding']
    single_score = score_complex(comp_single[i])['binding']
    print(f"Batch: {comp_batch[i]}, Binding Energy: {batch_score}")
    print(f"Single: {comp_single[i]}, Binding Energy: {single_score}")
    print(f"Difference (Batch - Single): {batch_score - single_score}\n")

Batch: ./batch_relax/id0_minimized.pdb, Binding Energy: -52.257239347804386
Single: ./single_relax/id0_minimized.pdb, Binding Energy: -37.26621531940617
Difference (Batch - Single): -14.991024028398215

Batch: ./batch_relax/id1_minimized.pdb, Binding Energy: -25.95194044697837
Single: ./single_relax/id1_minimized.pdb, Binding Energy: -18.42335476378289
Difference (Batch - Single): -7.52858568319548

Batch: ./batch_relax/id2_minimized.pdb, Binding Energy: -25.716733139269827
Single: ./single_relax/id2_minimized.pdb, Binding Energy: -25.359234167123937
Difference (Batch - Single): -0.35749897214589055

Batch: ./batch_relax/id3_minimized.pdb, Binding Energy: -36.32164961281779
Single: ./single_relax/id3_minimized.pdb, Binding Energy: -38.56847304414803
Difference (Batch - Single): 2.2468234313302418

Batch: ./batch_relax/id4_minimized.pdb, Binding Energy: -25.979728355553505
Single: ./single_relax/id4_minimized.pdb, Binding Energy: -27.547645764706886
Difference (Batch - Single): 1.567917